In [4]:
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration , Trainer,TrainingArguments


In [5]:
train_data=pd.read_csv("/content/data/samsum-train.csv")
validaqtion_data=pd.read_csv("/content/data/samsum-validation.csv")

In [6]:
#explore data
train_data.shape

(14732, 3)

In [7]:
validaqtion_data.shape

(818, 3)

In [8]:
print(train_data["dialogue"][2])
print("=============================================================================")
print(train_data["summary"][2])

Tim: Hi, what's up?
Kim: Bad mood tbh, I was going to do lots of stuff but ended up procrastinating
Tim: What did you plan on doing?
Kim: Oh you know, uni stuff and unfucking my room
Kim: Maybe tomorrow I'll move my ass and do everything
Kim: We were going to defrost a fridge so instead of shopping I'll eat some defrosted veggies
Tim: For doing stuff I recommend Pomodoro technique where u use breaks for doing chores
Tim: It really helps
Kim: thanks, maybe I'll do that
Tim: I also like using post-its in kaban style
Kim may try the pomodoro technique recommended by Tim to get more stuff done.


In [9]:
# اختيار 8000 سطر عشوائي للتدريب
small_train_data = train_data.sample(n=8000, random_state=42).reset_index(drop=True)

# اختيار 500 سطر عشوائي للاختبار (Validation)
small_val_data = validaqtion_data.sample(n=500, random_state=42).reset_index(drop=True)

In [10]:
small_train_data

,id,dialogue,summary
0,13811908,Violet: hi! i came across this Austin's articl...,Violet sent Claire Austin's article.
1,13716431,Pat: So does anyone know when the stream is go...,Pat and Lou are waiting for The stream but Kev...
2,13810214,Jane: <gif_file>\r\nJane: Whaddya think? \r\nS...,Jane is updating her Tinder profile tonight an...
3,13729823,"Adam: Do u have a map of Paris?\r\nTom: Yes, W...",Tom has a map of Paris.
4,13681400,"Frank: Hi, how's the family?\r\nMike: great! S...","Mike is happy, because Sam's moved out. Mike a..."
...,...,...,...
7995,13828491,Christa: Any news from Roger? He should be com...,Rita knows Roger is on a train home now and Ch...
7996,13680945,Bam: Hii :)\r\nKaty: Hey Hey :D\r\nBam: Can I ...,Bam asks Katy to go on a date on Friday at 7.30.
7997,13728987,Joe: Hello\r\nChelsea: Hey Joe. Longtime\r\nJo...,Chelsea will come to Joe's graduation 14th Dec...
7998,13682101,Susanne: hi\r\nUrsula: hello \r\nSusanne: how ...,Ursula is feeling low because her boyfriend ha...


In [11]:
small_val_data

,id,dialogue,summary
0,13680857,"Edd: wow, did you hear that they're transferri...",Rose and Edd will be transferred to a new depa...
1,13716124,"Tom: Where is the ""Sala del Capitolo""\r\nKevin...","""Sala del Capitolo"" Tom is looking for is in t..."
2,13864418,Patricia: The rowing practice is cancelled!\nK...,The rowing practice is cancelled. A few member...
3,13729340,"Tom: U OK?\r\nAlex: Yeah, pretty good. U?\r\nT...",Tom and Alex had fun last night. They drank a ...
4,13818813,"Patricia: Hello, here's the fair-trade brand I...",Patricia recommends a fair-trade brand she tal...
...,...,...,...
495,13820618,Izzy: Anyone knows where professor Xavier has ...,Professor Xavier probably holds his duty hours...
496,13680766,"Rita: didn't take breakfast with me, is there ...",Lina will give Rita one of her 2 sandwiches.
497,13819081,"Peter: Yo, we’re coming over to Warsaw for thi...",Peter and Jen are coming to Warsaw for the wee...
498,13862819,"Francis: Hey\nFrancis: Listen, I need a favor....",Francis asked Reynold for help with installing...


In [12]:
small_train_data["dialogue"][2]

"Jane: <gif_file>\r\nJane: Whaddya think? \r\nShona: This ur tinder profile thing?\r\nJane: Yeah, I'm updating my profile tonite. Kinda nervoous though... :( \r\nJane: What if i get another guy like John? o.O\r\nShona: John was a dickhead\r\nJane: preach sistah!\r\nShona: anyhoo - this time I've got u :D No slimeballs for you \r\nJane: Not again *shudders*\r\nJane: You know he forgot my birthday??!!\r\nShona: wanker"

In [13]:
#data cleaning
import re

def clean_text(text):
    text = re.sub(r'<.*?>', '', text)
    text = text.replace('\r\n', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

small_train_data["dialogue"] = small_train_data["dialogue"].apply(clean_text)
small_val_data["dialogue"] = small_val_data["dialogue"].apply(clean_text)
small_train_data["summary"] = small_train_data["summary"].apply(clean_text)
small_val_data["summary"] = small_val_data["summary"].apply(clean_text)

In [17]:
small_train_data["summary"][2]

"Jane is updating her Tinder profile tonight and together with Shona they don't want to find another guy like John, who forgot Jane's birthday."

In [16]:
small_train_data["dialogue"][2]

"Jane: Jane: Whaddya think? Shona: This ur tinder profile thing? Jane: Yeah, I'm updating my profile tonite. Kinda nervoous though... :( Jane: What if i get another guy like John? o.O Shona: John was a dickhead Jane: preach sistah! Shona: anyhoo - this time I've got u :D No slimeballs for you Jane: Not again *shudders* Jane: You know he forgot my birthday??!! Shona: wanker"

In [18]:
# load tokenizer
tokenizer=T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [45]:
from datasets import Dataset

# 1. تحويل Pandas DataFrame إلى Hugging Face Dataset
train_dataset = Dataset.from_pandas(small_train_data)
val_dataset = Dataset.from_pandas(small_val_data)

# 2. تجهيز دالة التقطيع (Tokenization)
def tokenize_function(examples):
    # إضافة كلمة summarize عشان موديل T5 يفهم المطلوب
    inputs = ["summarize: " + str(doc) for doc in examples["dialogue"]]

    # تحويل المحادثات لأرقام
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    # تحويل الملخصات لأرقام وتخزينها كـ Labels
    labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

# 3. تطبيق الدالة على البيانات كلها دفعة واحدة
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 8000
})

In [48]:
# 1. تحميل الموديل نفسه (T5-small)
model = T5ForConditionalGeneration.from_pretrained("t5-small")
from transformers import DataCollatorForSeq2Seq

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [50]:
# 2. إعدادات التدريب (الخطة اللي هيمشي عليها المدرب)
training_args = TrainingArguments(
    output_dir="./t5_summary_model", # الفولدر اللي هيتحفظ فيه الموديل بعد التدريب
    eval_strategy="epoch",     # الموديل هيمتحن نفسه على داتا الـ Validation بعد كل لفة
    learning_rate=2e-5,              # سرعة التعلم (رقم قياسي ومناسب جداً للـ Fine-tuning)
    per_device_train_batch_size=4,   # هياخد 4 محادثات يدرب عليهم في المرة الواحدة
    per_device_eval_batch_size=4,    # هياخد 4 محادثات يختبر بيهم
    num_train_epochs=3,              # عدد مرات التدريب على الداتا كلها (3 لفات ممتاز كبداية)
    weight_decay=0.01,               # بيمنع الموديل إنه يحفظ الداتا بدل ما يفهمها (Overfitting)
    save_total_limit=2,              # هيحفظ آخر نسختين بس عشان ميملاش مساحة كولاب
)

# 3. مجمع البيانات (Data Collator)
# ده الموظف اللي بيتأكد إن البيانات مترتبة صح ومحطوطة في شكل Tensors قبل ما تدخل للموديل
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 4. تجهيز المدرب (Trainer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,   # الـ 8000 سطر بتوعك
    eval_dataset=tokenized_val,      # الـ 500 سطر بتوع الاختبار
    data_collator=data_collator,
)

# 5. أمر بدء التدريب الفعلي
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.490592,0.433009
2,0.474253,0.420819
3,0.467748,0.418619


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6000, training_loss=0.5493216171264649, metrics={'train_runtime': 1657.3221, 'train_samples_per_second': 14.481, 'train_steps_per_second': 3.62, 'total_flos': 3248203235328000.0, 'train_loss': 0.5493216171264649, 'epoch': 3.0})

In [53]:
model.save_pretrained("t5_summarymodel")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [61]:
# 1. حفظ ملفات الـ Tokenizer في فولدر جديد
tokenizer.save_pretrained("./tokenizer_files")

# 2. ضغط الفولدر
!zip -r tokenizer_files.zip tokenizer_files/

# 3. أمر التحميل المباشر
from google.colab import files
files.download('tokenizer_files.zip')

  adding: tokenizer_files/ (stored 0%)
  adding: tokenizer_files/tokenizer.json (deflated 75%)
  adding: tokenizer_files/tokenizer_config.json (deflated 82%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [52]:
tokenizer.save_pretrained("tokenization_model")

('tokenization_model/tokenizer_config.json',
 'tokenization_model/tokenizer.json')

In [58]:
device = model.device

import re

def clean_text(text):
    text = re.sub(r"\r\n", " ", text) # Remove \r\n
    text = re.sub(r"\s+", " ", text) # Remove space
    text = re.sub(r"<.*?>", "", text) # Remove HTML tags
    text = text.strip().lower() # Chars to small

    return text

def summarize_dialogue(dialogue):
    dialogue = clean_text(dialogue)

    # التعديل الأول: إضافة الأمر الخاص بموديل T5
    dialogue = "summarize: " + dialogue

    inputs = tokenizer(dialogue, return_tensors = "pt", max_length = 512, padding = "max_length", truncation = True)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    outputs = model.generate(
        input_ids = inputs["input_ids"],          # التعديل الثاني: تصحيح input_ids
        attention_mask = inputs["attention_mask"], # التعديل الثالث: تصحيح الإملاء
        max_length = 175,
        num_beams = 4,
        early_stopping = True,
        length_penalty=2.0
    )

    # التعديل الرابع: إضافة سطر الترجمة عشان الدالة ترجعلك نص بدل مصفوفة أرقام
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return summary

In [59]:
sample_dialogue = """
John: Hey Sarah, did you get a chance to look at the new AI project proposal? \r\n
Sarah: Hi John! Yes, I read it this morning. <attachment_file> \r\n
John: What do you think? Are we ready to present it to the client tomorrow? \r\n
Sarah: The technical part is solid, but we need to simplify the budget section.
It has too many       extra spaces and complex terms. \r\n
John: Got it. I will fix the budget numbers and send you the updated version in an hour. \r\n
Sarah: Perfect. I'll review it and prepare the presentation slides.
"""

# استدعاء الدالة وطباعة الملخص
final_summary = summarize_dialogue(sample_dialogue)

print("الملخص النهائي:")
print(final_summary)

الملخص النهائي:
john has read the new ai project proposal this morning. he will present it tomorrow.
